In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
#Load datasets
customer=pd.read_csv("Zomato Data-Customer.csv")
restaurant=pd.read_csv("Zomato Data-Restaurants.csv")
orders=pd.read_csv("Zomato Data-Orders.csv")

In [8]:
#missing values
customer.isnull().sum()

Customer_id            0
Customer_name          0
City                   0
Signup_time            0
Acquisition_channel    0
dtype: int64

In [9]:
restaurant.isnull().sum()

restaurant_id      0
restaurant_name    0
cuisine            0
city               0
avg_rating         0
dtype: int64

In [10]:
orders.isnull().sum()

order_id           0
customer_id        0
restaurant_id      0
order_timestamp    0
order_amount       0
discount_amount    0
delivery_fee       0
payment_mode       0
order_status       0
dtype: int64

In [11]:
#Duplicate rows
customer.duplicated().sum()
restaurant.duplicated().sum()
orders.duplicated().sum()

np.int64(0)

In [12]:
print("Customer table shape: ",customer.shape)
print("Restaurant table shape: ",restaurant.shape)
print("Orders table shape: ",orders.shape)

Customer table shape:  (4999, 5)
Restaurant table shape:  (200, 5)
Orders table shape:  (50000, 9)


In [13]:
print(customer.dtypes,"\n")
print(restaurant.dtypes,"\n")
print(orders.dtypes)

Customer_id            object
Customer_name          object
City                   object
Signup_time            object
Acquisition_channel    object
dtype: object 

restaurant_id       object
restaurant_name     object
cuisine             object
city                object
avg_rating         float64
dtype: object 

order_id            object
customer_id         object
restaurant_id       object
order_timestamp     object
order_amount       float64
discount_amount    float64
delivery_fee         int64
payment_mode        object
order_status        object
dtype: object


In [14]:
#Convert date datatype
customer["Signup_time"] = pd.to_datetime(customer["Signup_time"],format="%d/%m/%Y")

In [15]:
orders["order_timestamp"] = pd.to_datetime(orders["order_timestamp"],format="%m/%d/%Y")

In [16]:
customer.info()
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Customer_id          4999 non-null   object        
 1   Customer_name        4999 non-null   object        
 2   City                 4999 non-null   object        
 3   Signup_time          4999 non-null   datetime64[ns]
 4   Acquisition_channel  4999 non-null   object        
dtypes: datetime64[ns](1), object(4)
memory usage: 195.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         50000 non-null  object        
 1   customer_id      50000 non-null  object        
 2   restaurant_id    50000 non-null  object        
 3   order_timestamp  50000 non-null  datetime64[ns]
 4   ord

In [17]:
#Check categorical values
orders["order_status"].value_counts()

order_status
Delivered    29837
Cancelled    10095
Refunded     10068
Name: count, dtype: int64

In [18]:
#Unique id
customer["Customer_id"].is_unique

True

In [19]:
restaurant["restaurant_id"].is_unique

True

In [20]:
orders["order_id"].is_unique

True

In [21]:
orders.describe()

,order_timestamp,order_amount,discount_amount,delivery_fee
count,50000,50000.000000,50000.000000,50000.000000
mean,2025-03-01 07:38:07.295999744,899.129451,36.276467,40.013940
min,2024-01-01 00:00:00,200.450000,0.000000,20.000000
25%,2024-08-01 00:00:00,574.345000,0.000000,30.000000
50%,2025-02-28 00:00:00,896.895000,0.000000,40.000000
75%,2025-10-02 00:00:00,1224.202500,74.182500,50.000000
max,2026-04-30 00:00:00,1598.970000,159.680000,60.000000
std,NaN,376.686808,50.260681,11.825142


In [22]:
#Feature engineering
customer["Signup_Year"] = customer["Signup_time"].dt.year
customer["Signup_Month"] = customer["Signup_time"].dt.month_name()

In [23]:
orders["order_year"] = orders["order_timestamp"].dt.year           #Yearly revenue trend
orders["order_month"] = orders["order_timestamp"].dt.month_name()  #Monthly sales trend
orders["order_month_number"] = orders["order_timestamp"].dt.month
orders["order_day"] = orders["order_timestamp"].dt.day_name()      #Weekday vs weekend analysis
orders["order_weekday"] = orders["order_timestamp"].dt.dayofweek

In [24]:
orders.head(3)

,order_id,customer_id,restaurant_id,order_timestamp,order_amount,discount_amount,delivery_fee,payment_mode,order_status,order_year,order_month,order_month_number,order_day,order_weekday
0,O000001,C03175,R0115,2025-09-28,1079.36,107.94,39,Cash,Refunded,2025,September,9,Sunday,6
1,O000002,C02432,R0077,2024-02-13,1499.63,149.96,42,Card,Delivered,2024,February,2,Tuesday,1
2,O000003,C01133,R0131,2024-03-07,597.18,59.72,22,Cash,Delivered,2024,March,3,Thursday,3


In [25]:
customer.columns = customer.columns.str.lower()

In [26]:
customer.head(3)

,customer_id,customer_name,city,signup_time,acquisition_channel,signup_year,signup_month
0,C00002,Rohan Agarwal,Delhi,2024-05-06,Organic,2024,May
1,C00003,Priya Verma,Hyderabad,2025-10-13,Instagram Ads,2025,October
2,C00004,Neha Singh,Mumbai,2024-02-06,WhatsApp Campaign,2024,February


In [27]:
!pip install pymysql sqlalchemy

Defaulting to user installation because normal site-packages is not writeable


In [28]:
#Connect to MySQL
from sqlalchemy import create_engine

username = "diksha"
password = "password"
host = "localhost"
port = "3306"
database = "zomato_analysis"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

print("Connected Successfully")

Connected Successfully


In [29]:
customer.to_sql("customer",con=engine,if_exists="replace",index=False)

4999

In [30]:
restaurant.to_sql("restaurant",con=engine,if_exists="replace",index=False)

200

In [31]:
orders.to_sql("orders",con=engine,if_exists="replace",index=False)

50000

In [32]:
import pandas as pd

pd.read_sql("SELECT * FROM customer LIMIT 5;", engine)

,customer_id,customer_name,city,signup_time,acquisition_channel,signup_year,signup_month
0,C00002,Rohan Agarwal,Delhi,2024-05-06,Organic,2024,May
1,C00003,Priya Verma,Hyderabad,2025-10-13,Instagram Ads,2025,October
2,C00004,Neha Singh,Mumbai,2024-02-06,WhatsApp Campaign,2024,February
3,C00005,Amit Kumar,Noida,2024-11-08,Referral,2024,November
4,C00006,Rohan Agarwal,Chennai,2024-06-04,Instagram Ads,2024,June


In [33]:
customer.to_csv("customer_clean.csv", index=False)
orders.to_csv("orders_clean.csv", index=False)
restaurant.to_csv("restaurant_clean.csv", index=False)